# S&P 500 Options: PatchTST

This notebook fits the declared PatchTST member of the sequence population snapshotted by
`09_deep_learning`. After publishing every PatchTST checkpoint, it verifies that the complete
NLinear, LSTM, and PatchTST population is present.

Prerequisites: `09_deep_learning` and `09a_lstm`.

**Why the population is declared in one notebook and filled by several.** The set of members is
a claim made once, before any of them is fitted, so that no family can be added or dropped after
its results are visible. This notebook fits the last declared member and then checks that all
three are present, which is the point at which the population becomes readable downstream.

## What this model is, and how it differs from the LSTM beside it

PatchTST cuts the lookback window into fixed-length patches, embeds each patch as a token, and
lets attention weigh every token against every other. Where the LSTM reads the window one
session at a time and carries a state forward, this model sees the whole window at once and
learns which parts of it to look at.

**What the difference buys.** Recurrence reaches a distant session only by carrying information
through every session between; attention reaches it directly, so a pattern that depends on two
separated stretches of the window is easier to represent. It also parallelizes across the
window, where recurrence is sequential by construction.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# S&P 500 選擇權：PatchTST

本 Notebook 擬合 `09_deep_learning` 所建立快照中，已宣告的 PatchTST 序列模型成員。發布所有 PatchTST 檢查點後，會驗證完整的 NLinear、LSTM 與 PatchTST 模型群是否到齊。

先備條件：`09_deep_learning` 與 `09a_lstm`。

**為何在一份 Notebook 宣告模型群，卻由多份 Notebook 填入。** 成員集合只在擬合任何成員前宣告一次，使任何模型家族都無法在看見結果後新增或移除。本 Notebook 擬合最後一個已宣告成員，再檢查三個成員是否全部存在；此時模型群才可供下游讀取。

## 模型是什麼，以及與旁邊的 LSTM 有何不同

PatchTST 將回看視窗切成固定長度的片段，將每段嵌入為一個 token，再以注意力機制衡量各 token 之間的關係。LSTM 每次讀取視窗中的一個交易日，並把狀態向前傳；此模型則一次看到完整視窗，學習應關注其中哪些部分。

**差異帶來什麼好處。** 循環模型若要利用久遠交易日的資訊，須把資訊沿中間每個交易日一路傳遞；注意力機制則可直接連接，因此更容易表示依賴視窗中兩段相隔期間的型態。它還可在視窗上平行運算，而循環模型依結構必須依序運算。
<!-- ML4T_ZH_END c0000-p000 -->

**What it costs, and why both are run.** Attention has no built-in notion that yesterday is
nearer than last month - the ordering has to be learned from position embeddings rather than
being structural - and it has more parameters to fit from the same data. Running both against
the same population and the same folds is what turns "sequence models on options data" from an
assertion into a comparison, and either can lose.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**代價是什麼，以及為何兩者都要執行。** 注意力機制本身不知道昨日比上個月更近；順序須從位置嵌入中學得，而非由結構直接提供。它也需要從相同資料中擬合更多參數。讓兩者在相同模型群與資料折上執行，才能把「選擇權資料上的序列模型」從主張變成比較；任何一方都可能落敗。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""Fit the declared S&P 500 options PatchTST request."""

import polars as pl

from case_studies.sp500_options.research_workflow import (
    ALL_LABELS,
    declared_dl_device,
    model_request_catalog,
    open_study,
    published_dl_device,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_subset,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
DEVICE: str = ""

POPULATION_NAME: str = ""

### The device the population was fitted on

A network trained on a GPU and the same network trained on a CPU accumulate their sums in a
different order and reach different weights, so the device is part of what the fitted model is
and sits inside the training identity rather than beside it. The device this population was
fitted on is declared once, in `modeling.dl.device` in `config/setup.yaml`, and read from there
by all four deep-learning notebooks rather than retyped in each. On a machine with no NVIDIA
card the run stops here rather than quietly training something else: set `DEVICE="cpu"` and pass
a `POPULATION_NAME` to fit the same requests there, under a name of their own.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
### 模型群擬合時使用的裝置

相同網路分別在 GPU 與 CPU 上訓練，加總順序不同，最終權重也可能不同，因此運算裝置屬於已擬合模型定義的一部分，應記入訓練身分，而非只放在旁邊。本模型群使用的裝置，透過 `modeling.dl.device` 於 `config/setup.yaml` 中宣告一次，由四份深度學習 Notebook 共同讀取，不各自重填。若電腦沒有 NVIDIA 顯示卡，執行會在此停止，而非悄悄訓練不同模型；可設定 `DEVICE="cpu"` 並傳入 `POPULATION_NAME`，在該裝置上以自身名稱擬合相同請求。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
CANONICAL_POPULATION_NAME = "sp500-options-sequence-validation-v1"

published_device = published_dl_device()
device = declared_dl_device(DEVICE)
population_name = POPULATION_NAME or CANONICAL_POPULATION_NAME
if device != published_device and population_name == CANONICAL_POPULATION_NAME:
    raise ValueError(
        f"this run fits on {device!r}, not the published {published_device!r}, so its "
        f"identities are not the ones {CANONICAL_POPULATION_NAME!r} holds; pass "
        f"POPULATION_NAME to give them a population of their own"
    )
print(f"training device: {device} (declared: {published_device})")

training device: cuda (declared: cuda)


## Declared request

**What the settings decide.** `lookback: 60` and `patch_size: 16` together set the tokens: a
sixty-session window becomes a handful of patches rather than sixty steps, which is what makes
attention affordable here and also what limits its resolution, since nothing inside a patch is
distinguished. `d_model: 64` is the width each patch is embedded into and `n_heads: 4` the
number of attention patterns learned in parallel, so the model can attend to several
relationships at once instead of averaging them into one. `n_layers: 2` stacks that twice.
`dropout: 0.1` is the same regularizer the LSTM uses, and deliberately so: two families whose
regularization differs are not being compared on architecture.

**The configuration is read from a preset, not written here**, so this architecture cannot
quietly differ from the same architecture in another chapter.

**Every label is fitted**, because selection downstream ranks across labels as well as across
configurations, and a label with no candidates cannot be chosen or ruled out.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 已宣告的請求

**各設定決定什麼。** `lookback: 60` 與 `patch_size: 16` 共同決定 token：六十交易日的視窗不再是六十步，而是少數片段，使此處的注意力運算可負擔；同時也限制解析度，因為同一片段內的差異不再區分。`d_model: 64` 是各片段嵌入後的寬度，`n_heads: 4` 是平行學習的注意力型態數量，使模型能同時關注多種關係，而非全部平均成一種。`n_layers: 2` 將這種結構堆疊兩次。`dropout: 0.1` 刻意採用與 LSTM 相同的正則化方法；若兩個模型家族的正則化不同，比較的便不只是架構。

**設定由預設檔讀取，而非寫死於此**，避免此架構與其他章節中的相同架構悄悄不同。

**會擬合所有標籤**，因為下游會跨標籤與設定一起排名；若某標籤沒有候選模型，就無法選用，也無法排除。
<!-- ML4T_ZH_END c0005-p000 -->


In [4]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
requests = model_request_catalog(
    "deep_learning",
    labels=ALL_LABELS,
    config_names=("patchtst",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""ret_to_expiry""","""patchtst""","""regression""",54,273,85083,2,2019-01-07 00:00:00,2020-11-25 00:00:00,20,"""canonical""","""10506ca49b0c"""


## Execute and validate

The shared sequence runner owns gap-safe window construction, fold fitting, fitted-state reload,
checkpoint publication, restart, and exact eligible-key validation.

**A checkpoint is part of a configuration, not a detail of how it was fitted.** Training runs for
100 epochs and publishes every fifth, so this one request becomes twenty scored candidates. A
network's validation performance is not monotone in training time, and the epoch at which it
peaks is a property of the fit a reader is entitled to see rather than a number chosen after the
fact. Picking the best epoch after seeing the results is selection, and selection happens once,
downstream, on backtests.

**Gap-safe means a window never reaches across a fold boundary.** A sequence handed to the model
has to end before the validation window opens, or its state carries information from the period
being scored. The runner owns that construction because it is exactly the kind of rule that gets
restated slightly differently in each notebook and is impossible to notice when it is.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行與驗證

共用序列模型執行器負責可安全處理缺口的視窗建構、資料折擬合、重新載入已擬合狀態、發布檢查點、重新啟動，以及精確驗證符合條件的鍵值。

**檢查點是設定的一部分，而非擬合過程的附帶細節。** 訓練共進行 100 個 epoch，每五個發布一個檢查點，因此單一請求會變成二十個受評分候選模型。神經網路的驗證表現不會隨訓練時間單調變化；達到高峰的 epoch 是擬合結果本身的性質，讀者應能看見，而非事後挑選的數字。看過結果後挑選最佳 epoch 屬於選擇，只在下游依回測結果進行一次。

**安全處理缺口，表示視窗絕不跨越資料折邊界。** 交給模型的序列必須在驗證視窗開始前結束，否則狀態會帶入受評分期間的資訊。執行器統一負責建構，因為若各 Notebook 分別重述此規則，容易出現細微差異，也很難察覺。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_subset(
        study,
        resolved,
        population=population_name,
        require_population_complete=True,
    )
else:
    if not WORKSPACE or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("PatchTST execution returned a partial checkpoint")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",5,"""canonical""",true,"""10506ca49b0c""","""2c182fd08460"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",10,"""canonical""",true,"""10506ca49b0c""","""3ba60ad659a9"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",15,"""canonical""",true,"""10506ca49b0c""","""2202f5ac7c8c"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",20,"""canonical""",true,"""10506ca49b0c""","""7b07b18533d3"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",25,"""canonical""",true,"""10506ca49b0c""","""de7bb42271d4"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",80,"""canonical""",true,"""10506ca49b0c""","""f32884aefbdf"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",85,"""canonical""",true,"""10506ca49b0c""","""f08ef4b051de"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",90,"""canonical""",true,"""10506ca49b0c""","""f73f1f4d34c4"""


The official sequence population is complete and ready for model analysis and backtesting. This
notebook does not compare configurations or choose a checkpoint.

**What completeness means here, and why it is checked rather than assumed.** Every requested
checkpoint of all three families produced predictions on exactly the rows its eligibility
contract declared, not more and not fewer. A partial checkpoint is refused rather than
published: a downstream comparison against a model scored on part of the panel is not a
comparison, and by the time anyone reads the result the missing part is invisible.

**The three families share one eligibility group, and that is what makes them comparable.** All
of them need the same sixty-session window before a symbol can be scored, so they are eligible
on the same rows and the difference between their numbers is the models. That does not extend to
the cross-sectional families, which score a symbol from its first row; `11_model_analysis` groups
by eligibility for exactly that reason.

**What has and has not been established at this point.** Three architectures have been fitted on
identical folds, identical windows and identical labels, and every checkpoint each of them
published is registered and complete. Nothing has been ranked. A reader who wants to know which
sequence family works better on options data has the material for that question, and the answer
comes from backtests rather than from anything on this page.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
正式序列模型群已完整，可供模型分析與回測。本 Notebook 不比較設定，也不選擇檢查點。

**此處的完整性代表什麼，以及為何要檢查、不能直接假設。** 三個模型家族要求的每個檢查點，恰好在資格規則宣告的資料列上產生預測，不多也不少。部分完成的檢查點會被拒絕，而非發布；若下游拿只在資料面板部分範圍評分的模型比較，就不是有效比較，且讀取結果時缺失部分通常已不可見。

**三個模型家族共用同一資格組別，因而可以比較。** 每個模型都需要同一段六十交易日視窗，才可對某檔股票評分，因此符合條件的資料列相同，數值差異才來自模型。橫斷面模型家族則不同，它從股票首列就能評分；`11_model_analysis` 正因此依資格分組。

**目前確立了什麼，還沒有確立什麼。** 三種架構已在相同資料折、視窗與標籤上擬合，且各自發布的所有檢查點都完整登錄。尚未進行排名。想知道哪種序列模型家族在選擇權資料上表現較佳的讀者，現在已有回答所需資料，但答案來自回測，而非本頁的內容。
<!-- ML4T_ZH_END c0010-p000 -->
